# How many neighbouring samples belong in a slope estimate?

A wider local fit suppresses measurement noise, yet it can smooth away real changes in slope. Derive the competing errors, then use noisy synthetic data to find which differentiator wins at a chosen spacing.

## Fit a polynomial in local coordinates

Suppose we observe $y_j=f(jh)+\epsilon_j$ at equally spaced times around zero. The errors are independent with mean zero and common standard deviation $\sigma$. Fit $a+bx+cx^2$ by minimizing the sum of squared differences on five points with $j=-2,-1,0,1,2$. Differentiate that sum with respect to $b$. Because $\sum j=\sum j^3=0$, the odd column of the design matrix is orthogonal to both even columns. The fitted slope is therefore

$$\hat b=\frac{\sum j y_j}{h\sum j^2}
        =\frac{-2y_{-2}-y_{-1}+y_1+2y_2}{10h}.$$

Its weights have zero sum and give exactly one on a unit linear ramp. A quadratic term cancels by symmetry. A cubic term $\beta x^3$ contributes $\beta h^2\sum j^4/\sum j^2=3.4\beta h^2$ to the estimate. Meanwhile the independent noise variances add after weighting. Their sum is $\sigma^2/(10h^2)$. For a three-point fit the same calculation gives a cubic bias of $\beta h^2$ and a variance of $\sigma^2/(2h^2)$.

Write the leading bias as $Bh^2$ and the noise standard deviation as $S/h$. The mean square error around the true derivative is $B^2h^4+S^2/h^2$. Its minimum is at $h=(S^2/(2B^2))^{1/6}$ when both contributions are present. Repeated noisy measurements could reduce the noise term too. Correlated measurement errors require the covariance matrix between samples in the variance calculation.

## Diagnose a surprising result at two spacings

Use a signal $f(x)=1+x+0.4x^2+x^3$ and noise standard deviation 0.02. The target derivative at zero is one. At $h=0.20$ the five-point bias is 0.136 and its noise standard deviation is about 0.0316, giving a root mean square error of 0.1396. The three-point bias is 0.040, its standard deviation is 0.0707 and its total error is 0.0812. The shorter fit wins despite its greater sensitivity to random noise.

At $h=0.08$ the five-point values change to 0.02176 and 0.07906, giving a total of 0.0820. The three-point result is about 0.1769. Changing the spacing has exchanged the dominant source of error. A single realization can put its estimate on either side of the derivative. Change the random seed while keeping the spacing fixed to distinguish this variability from the average error in the second plot.

## Set the experimental inputs

Edit these values, then run the calculation and both figures again.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

teal, orange, blue = "#19596a", "#aa462a", "#3868a0"
def axes(xlabel, ylabel):
    fig, ax = plt.subplots(figsize=(7, 4), constrained_layout=True)
    ax.set(xlabel=xlabel, ylabel=ylabel)
    ax.grid(alpha=0.2)
    return fig, ax

In [ ]:
spacing = 0.20
noise_sd = 0.02
cubic = 1.0
seed = 17

In [ ]:
assert spacing > 0 and noise_sd >= 0
def signal(x):
    return 1 + x + .4*x*x + cubic*x**3

def slope_statistics(radius,h):
    j = np.arange(-radius,radius+1,dtype=float)
    j2 = np.dot(j,j)
    bias = cubic*h*h*np.sum(j**4)/j2
    noise = noise_sd/(h*np.sqrt(j2))
    return bias,noise,np.hypot(bias,noise)

rng = np.random.default_rng(seed)
j = np.arange(-2,3,dtype=float)
x = spacing*j
y = signal(x)+rng.normal(0,noise_sd,size=5)
models = {}
for radius in (1,2):
    keep = np.abs(j)<=radius
    coef = np.polynomial.polynomial.polyfit(x[keep],y[keep],2)
    weights = j[keep]/(spacing*np.dot(j[keep],j[keep]))
    assert np.isclose(coef[1],weights @ y[keep])
    bias,noise,rmse = slope_statistics(radius,spacing)
    assert np.isclose(weights @ signal(x[keep]),1+bias)
    models[radius] = coef
    print(f"{2*radius+1} samples: slope {coef[1]:.5f}, bias {bias:.5f}, noise SD {noise:.5f}, RMSE {rmse:.5f}")

In [ ]:
fig, ax = axes("Time relative to fit centre", "Sensor signal")
dense = np.linspace(-2.2*spacing,2.2*spacing,200)
ax.plot(dense,signal(dense),color="#414141",label="Exact signal")
ax.errorbar(x,y,yerr=noise_sd,fmt="o",color="black",capsize=3,label="Samples ± one SD")
for r,color in [(1,orange),(2,teal)]:
    local = dense[np.abs(dense)<=r*spacing]
    ax.plot(local,np.polynomial.polynomial.polyval(local,models[r]),ls="--",color=color,
            label=f"{2*r+1}-point fit, slope {models[r][1]:.3f}")
ax.legend(fontsize=8)
plt.show()

The three-point and five-point local fits use nested samples. Inspect the tangent at the centre, where the derivative of the underlying cubic signal equals one.

In [ ]:
fig, ax = axes("Sample interval h", "Root mean square slope error")
hs = spacing*np.geomspace(.08,5,150)
for r,color in [(1,orange),(2,teal)]:
    err = slope_statistics(r,hs)[2]
    ax.plot(hs,err,color=color,label=f"{2*r+1} points")
    ax.scatter(spacing,slope_statistics(r,spacing)[2],color=color)
ax.set_xscale("log")
if noise_sd > 0 or cubic != 0: ax.set_yscale("log")
ax.legend()
plt.show()

For a cubic signal the bias-plus-variance formula is exact. The location of the minimum is an experimental design choice controlled by signal curvature change and noise.

## Exercise · predict a single perturbation

A five-point experiment at spacing 0.2 measures the rightmost sample 0.03 above its true value. By how much does that change the slope? What happens if the same offset appears on all five samples?

The rightmost derivative weight is $2/(10h)=1$ in these units. The slope rises by 0.03. A common offset contributes its magnitude times the sum of all weights, which is zero. This cancellation also explains why an uncertain but constant instrument zero does not affect this derivative.

## Exercise · choose a spacing when noise increases

If an otherwise identical experiment has eight times the noise standard deviation, how should the optimum spacing change? Will this scaling hold for a perfectly quadratic signal?

With fixed cubic coefficient $B$, the optimum is proportional to $\sigma^{1/3}$. Eight times the standard deviation therefore doubles the best spacing. For a quadratic the cubic bias vanishes identically. The stated model then predicts decreasing noise error as the spacing grows. In a real record the usable window can be limited by unmodelled higher derivatives, nearby boundaries or changing noise statistics.